# Add new parameters and constraints in ES

In [2]:
import pandas as pd
from shared.utils import load_snapshot
from energyscope.models import Model

YEAR: year of optimization
TEC: technology
MAT: material

Parameters:
- `material_intensity[YEAR, TEC, MAT]` [kt / GW]
- `limit_material_year[YEAR, MAT]` [kt]
- `limit_material[MAT]` [kt]
- `recycling_rate[YEAR, (TEC), MAT]` [-]

Variables:
- `Material_content_year[YEAR, TEC, MAT]` [kt]
- `Material_content[TEC, MAT]` [kt]
- `Recycled_material[YEAR, TEC, MAT]` [kt]

Constraints:
- `Material_content_year[YEAR, TEC, MAT] = material_intensity[YEAR, TEC, MAT] * F_new[YEAR, TEC, MAT]`
- ...

In [2]:
data = [
    ['material_intensity', 'YEAR_2020', 'WIND_ONSHORE', 'Li', 0.0013, 'kt/GW', 'a comment'],
]
df = pd.DataFrame(data, columns=['Parameter', 'index0', 'index1', 'index2', 'Value', 'Unit', 'Comment'])

In [6]:
df = pd.read_excel("excel_files/technologies_mi_all_years.xlsx")

# Vérifier que ça a bien été lu
print(df.head())
print(df.shape)
print(df.dtypes)

            Parameter index0 index1     index2    Value  Unit  Comment
0  material_intensity    AFC     Al  YEAR_2020  121.125  t/GW      NaN
1  material_intensity    AFC     Al  YEAR_2025  121.125  t/GW      NaN
2  material_intensity    AFC     Al  YEAR_2030  121.125  t/GW      NaN
3  material_intensity    AFC     Al  YEAR_2035  121.125  t/GW      NaN
4  material_intensity    AFC     Al  YEAR_2040  121.125  t/GW      NaN
(174132, 7)
Parameter     object
index0        object
index1        object
index2        object
Value        float64
Unit          object
Comment      float64
dtype: object


In [9]:
def create_dat_file_from_excel(df, file_name):
    out_path = f'ampl_files/{file_name}.dat'
    # use utf-8-sig so Windows Notepad shows accents correctly; use 'utf-8' if BOM is not desired
    with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
        f.write("set MATERIALS := Al B Cd Cr Co Concrete Cu Dy Ga Glass Ge Hf In Fe Pb Polymers Li Mg Mn Mo Nd Ni Nb Pr Se Si Ag Ta Te Tb Sn W V Y Zn Zr ;\n \n")
        for _, row in df.iterrows():
            param_name = row['Parameter']
            index0 = row['index0']
            index1 = row['index1']
            index2 = row['index2']
            unit = '-' if pd.isna(row.get('Unit')) else str(row.get('Unit'))
            value = row['Value']
            comment = '' if pd.isna(row.get('Comment')) else str(row.get('Comment'))
            if pd.isna(index1) and pd.isna(index2):
                f.write(f"let {param_name}['{index0}'] := {value} ; # [{unit}] {comment}\n")
            elif pd.isna(index2):
                f.write(f"let {param_name}['{index0}','{index1}'] := {value} ; # [{unit}] {comment}\n")
            else:
                f.write(f"let {param_name}['{index0}','{index1}','{index2}'] := {value} ; # [{unit}] {comment}\n")

In [ ]:
create_dat_file_from_excel(df, 'Material_intensity')

In [14]:
# Add your files to the main model
main_model = load_snapshot(2050) + Model([
    ('mod', 'ampl_files/test.mod'),
    ('dat', 'ampl_files/test.dat'),
])